# ROGII - DWT-Inspired Modeling

Advanced Modeling V2 improved the public score to `15.049`, while V3 dropped to `15.306`. This notebook starts a new candidate inspired by DWT-based log-shape matching.

The idea is to compare horizontal-well `GR` and typewell `GR` at multiple scales. Instead of only asking whether two raw `GR` values match at one row, the notebook builds Haar-style detail features that capture local changes over short, medium, and long windows.

Workflow:

1. Keep carry-forward as the safety anchor.
2. Build standard rolling features from the best baseline family.
3. Add typewell-offset features plus multi-scale Haar/DWT-like `GR` detail differences.
4. Train a residual tree model under held-out-well masked-tail validation.
5. Generate `submission.csv` only when validation beats carry-forward.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 160)
pd.set_option('display.max_rows', 80)

RANDOM_STATE = 42
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
COMPETITION_SLUG = 'rogii-wellbore-geology-prediction'
DATA_ROOT_CANDIDATES = [
    KAGGLE_INPUT_ROOT / 'competitions' / COMPETITION_SLUG,
    KAGGLE_INPUT_ROOT / COMPETITION_SLUG,
]
WORK_DIR = Path('/kaggle/working')
SUBMISSION_PATH = WORK_DIR / 'submission.csv'


def resolve_data_root(candidates):
    for candidate in candidates:
        if (candidate / 'sample_submission.csv').exists():
            return candidate
    for sample_file in KAGGLE_INPUT_ROOT.rglob('sample_submission.csv') if KAGGLE_INPUT_ROOT.exists() else []:
        if COMPETITION_SLUG in sample_file.as_posix():
            return sample_file.parent
    return candidates[0]


DATA_ROOT = resolve_data_root(DATA_ROOT_CANDIDATES)
print('DATA_ROOT:', DATA_ROOT)
print('sample_submission exists:', (DATA_ROOT / 'sample_submission.csv').exists())


## 1. Load Files

The notebook uses horizontal wells, matching typewells, and `sample_submission.csv`. During validation, training wells are masked to simulate the public hidden interval.


In [ ]:
def find_files(root: Path, pattern: str):
    return sorted(root.rglob(pattern)) if root.exists() else []


def well_name_from_horizontal_path(path: Path) -> str:
    return path.name.split('__horizontal_well.csv')[0]


def well_name_from_typewell_path(path: Path) -> str:
    return path.name.split('__typewell.csv')[0]


def parse_submission_id(value):
    well, row = str(value).rsplit('_', 1)
    return well, int(row)


def get_column(df: pd.DataFrame, name: str):
    lookup = {col.lower(): col for col in df.columns}
    return lookup.get(name.lower())


train_files = find_files(DATA_ROOT / 'train', '*__horizontal_well.csv')
test_files = find_files(DATA_ROOT / 'test', '*__horizontal_well.csv')
train_typewell_files = find_files(DATA_ROOT / 'train', '*__typewell.csv')
test_typewell_files = find_files(DATA_ROOT / 'test', '*__typewell.csv')

train_typewell_lookup = {well_name_from_typewell_path(path): path for path in train_typewell_files}
test_typewell_lookup = {well_name_from_typewell_path(path): path for path in test_typewell_files}

sample_submission = pd.read_csv(DATA_ROOT / 'sample_submission.csv')
id_col = sample_submission.columns[0]
target_col = 'tvt' if 'tvt' in sample_submission.columns else sample_submission.columns[-1]
parsed_ids = sample_submission[id_col].map(parse_submission_id)
sample_submission['well'] = [item[0] for item in parsed_ids]
sample_submission['row_idx'] = [item[1] for item in parsed_ids]

print('train horizontal wells:', len(train_files))
print('train typewells:', len(train_typewell_files))
print('test horizontal wells:', len(test_files))
print('test typewells:', len(test_typewell_files))
print('submission rows:', len(sample_submission))
display(sample_submission.head())


## 2. Feature Engineering

The DWT-inspired layer uses Haar-style details:

`detail(scale) = rolling_mean(current scale) - rolling_mean(previous scale)`

This approximates wavelet detail coefficients while staying fast and dependency-free in Kaggle. The model receives:

- base rolling `GR` and `TVT_input` features;
- typewell `GR` at candidate `TVT` offsets;
- multi-scale detail differences between horizontal `GR` and typewell `GR`;
- best offset and best detail-match features per scale.


In [ ]:
ROLL_WINDOWS = (25, 101, 301)
TYPEWELL_OFFSETS = np.array([-200.0, -100.0, -50.0, 0.0, 50.0, 100.0, 200.0], dtype='float64')
DWT_SCALES = (16, 64, 256)
RESIDUAL_SHRINKAGES = (0.70, 0.85, 1.00)
FEATURE_COLUMNS = None
NON_FEATURE_COLUMNS = {'well', 'target_tvt', 'target_residual', 'tail_fraction'}
TYPEWELL_CACHE = {}


def numeric_col(df, name, default=np.nan):
    col = get_column(df, name)
    if col is None:
        return pd.Series(default, index=df.index, dtype='float64')
    return pd.to_numeric(df[col], errors='coerce').astype('float64')


def tvt_input_series(df):
    tvt_input_col = get_column(df, 'TVT_input')
    tvt_col = get_column(df, 'TVT')
    if tvt_input_col is not None:
        return pd.to_numeric(df[tvt_input_col], errors='coerce').astype('float64').reset_index(drop=True)
    if tvt_col is not None:
        return pd.to_numeric(df[tvt_col], errors='coerce').astype('float64').reset_index(drop=True)
    return pd.Series(np.nan, index=range(len(df)), dtype='float64')


def carry_forward_prediction(df):
    y_input = tvt_input_series(df)
    carry = y_input.ffill().bfill()
    if carry.isna().all():
        carry = pd.Series(0.0, index=range(len(df)), dtype='float64')
    return carry.astype('float64')


def add_rolling_features(out, source, prefix):
    for window in ROLL_WINDOWS:
        rolled = source.rolling(window=window, min_periods=1)
        out[f'{prefix}_roll_mean_{window}'] = rolled.mean()
        out[f'{prefix}_roll_std_{window}'] = rolled.std().fillna(0.0)
        out[f'{prefix}_roll_min_{window}'] = rolled.min()
        out[f'{prefix}_roll_max_{window}'] = rolled.max()
    return out


def haar_detail(values, scale):
    series = pd.Series(values, dtype='float64')
    recent = series.rolling(window=scale, min_periods=max(2, scale // 4)).mean()
    previous = series.shift(scale).rolling(window=scale, min_periods=max(2, scale // 4)).mean()
    return (recent - previous).replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype='float64')


def offset_label(offset):
    return f'{int(offset):+d}'


def clean_numeric_frame(out):
    numeric_features = [col for col in out.columns if col != 'well']
    out[numeric_features] = out[numeric_features].replace([np.inf, -np.inf], np.nan)
    medians = out[numeric_features].median(numeric_only=True)
    out[numeric_features] = out[numeric_features].fillna(medians).fillna(0.0)
    return out


def load_typewell_curve(path):
    if path is None:
        return None
    cache_key = str(path)
    if cache_key in TYPEWELL_CACHE:
        return TYPEWELL_CACHE[cache_key]

    df = pd.read_csv(path)
    tvt_col = get_column(df, 'TVT')
    gr_col = get_column(df, 'GR')
    if tvt_col is None or gr_col is None:
        TYPEWELL_CACHE[cache_key] = None
        return None

    curve = pd.DataFrame({
        'tvt': pd.to_numeric(df[tvt_col], errors='coerce'),
        'gr': pd.to_numeric(df[gr_col], errors='coerce'),
    }).dropna()
    curve = curve.sort_values('tvt').drop_duplicates('tvt')
    if len(curve) < 2:
        TYPEWELL_CACHE[cache_key] = None
        return None

    gr = curve['gr'].interpolate(limit_direction='both').to_numpy(dtype='float64')
    tvt = curve['tvt'].to_numpy(dtype='float64')
    TYPEWELL_CACHE[cache_key] = {'tvt': tvt, 'gr': gr}
    return TYPEWELL_CACHE[cache_key]


def interpolate_typewell_gr(curve, tvt_values):
    if curve is None:
        return np.full(len(tvt_values), np.nan, dtype='float64')
    return np.interp(tvt_values, curve['tvt'], curve['gr'], left=curve['gr'][0], right=curve['gr'][-1])


def add_empty_typewell_dwt_features(out):
    for offset in TYPEWELL_OFFSETS:
        label = offset_label(offset)
        out[f'typewell_gr_offset_{label}'] = 0.0
        out[f'typewell_absdiff_offset_{label}'] = 0.0
        for scale in DWT_SCALES:
            out[f'dwt_absdiff_s{scale}_offset_{label}'] = 0.0
            out[f'dwt_signeddiff_s{scale}_offset_{label}'] = 0.0
    for scale in DWT_SCALES:
        out[f'horizontal_dwt_detail_s{scale}'] = 0.0
        out[f'dwt_best_absdiff_s{scale}'] = 0.0
        out[f'dwt_best_offset_s{scale}'] = 0.0
    out['typewell_available'] = 0
    out['typewell_gr_at_carry'] = 0.0
    out['typewell_gr_diff_at_carry'] = 0.0
    out['typewell_best_offset_raw_gr'] = 0.0
    out['typewell_best_absdiff_raw_gr'] = 0.0
    out['dwt_multiscale_best_score'] = 0.0
    out['dwt_multiscale_best_offset'] = 0.0
    return out


def add_typewell_dwt_features(out, gr_interp, carry, typewell_path):
    curve = load_typewell_curve(typewell_path)
    carry_values = carry.to_numpy(dtype='float64')
    gr_values = gr_interp.to_numpy(dtype='float64')

    horizontal_details = {scale: haar_detail(gr_values, scale) for scale in DWT_SCALES}
    for scale, detail in horizontal_details.items():
        out[f'horizontal_dwt_detail_s{scale}'] = detail

    if curve is None:
        return add_empty_typewell_dwt_features(out)

    candidate_gr = []
    multiscale_scores = []
    detail_diffs_by_scale = {scale: [] for scale in DWT_SCALES}

    for offset in TYPEWELL_OFFSETS:
        label = offset_label(offset)
        aligned_gr = interpolate_typewell_gr(curve, carry_values + offset)
        candidate_gr.append(aligned_gr)
        out[f'typewell_gr_offset_{label}'] = aligned_gr
        out[f'typewell_absdiff_offset_{label}'] = np.abs(gr_values - aligned_gr)

        offset_score_parts = []
        for scale in DWT_SCALES:
            type_detail = haar_detail(aligned_gr, scale)
            signed_diff = horizontal_details[scale] - type_detail
            abs_diff = np.abs(signed_diff)
            detail_diffs_by_scale[scale].append(abs_diff)
            offset_score_parts.append(abs_diff)
            out[f'dwt_absdiff_s{scale}_offset_{label}'] = abs_diff
            out[f'dwt_signeddiff_s{scale}_offset_{label}'] = signed_diff
        multiscale_scores.append(np.mean(np.vstack(offset_score_parts), axis=0))

    candidate_gr = np.vstack(candidate_gr).T
    raw_diffs = np.abs(candidate_gr - gr_values[:, None])
    raw_best_idx = np.nanargmin(raw_diffs, axis=1)
    rows = np.arange(len(out))

    out['typewell_available'] = 1
    zero_offset_idx = int(np.where(TYPEWELL_OFFSETS == 0.0)[0][0])
    out['typewell_gr_at_carry'] = candidate_gr[:, zero_offset_idx]
    out['typewell_gr_diff_at_carry'] = gr_values - out['typewell_gr_at_carry']
    out['typewell_best_offset_raw_gr'] = TYPEWELL_OFFSETS[raw_best_idx]
    out['typewell_best_absdiff_raw_gr'] = raw_diffs[rows, raw_best_idx]

    for scale, diff_list in detail_diffs_by_scale.items():
        diff_matrix = np.vstack(diff_list).T
        best_idx = np.nanargmin(diff_matrix, axis=1)
        out[f'dwt_best_absdiff_s{scale}'] = diff_matrix[rows, best_idx]
        out[f'dwt_best_offset_s{scale}'] = TYPEWELL_OFFSETS[best_idx]

    score_matrix = np.vstack(multiscale_scores).T
    best_score_idx = np.nanargmin(score_matrix, axis=1)
    out['dwt_multiscale_best_score'] = score_matrix[rows, best_score_idx]
    out['dwt_multiscale_best_offset'] = TYPEWELL_OFFSETS[best_score_idx]
    return out


def build_features(df, well, typewell_lookup):
    n = len(df)
    idx = pd.Series(np.arange(n), dtype='float64')
    denom = max(n - 1, 1)

    md = numeric_col(df, 'MD').reset_index(drop=True)
    x = numeric_col(df, 'X').reset_index(drop=True)
    y = numeric_col(df, 'Y').reset_index(drop=True)
    z = numeric_col(df, 'Z').reset_index(drop=True)
    gr_raw = numeric_col(df, 'GR').reset_index(drop=True)
    y_input = tvt_input_series(df)
    carry = carry_forward_prediction(df)

    gr_interp = gr_raw.interpolate(limit_direction='both').ffill().bfill()
    if gr_interp.isna().all():
        gr_interp = pd.Series(0.0, index=range(n), dtype='float64')

    known = y_input.notna()
    known_idx = pd.Series(np.where(known, idx, np.nan)).ffill().fillna(0.0)
    distance_from_known = (idx - known_idx).clip(lower=0)
    hidden_flag = (~known).astype('int8')

    tvt_diff = carry.diff().fillna(0.0)
    recent_slope = tvt_diff.rolling(window=101, min_periods=1).mean().fillna(0.0)
    recent_volatility = tvt_diff.rolling(window=101, min_periods=1).std().fillna(0.0)

    out = pd.DataFrame({
        'well': well,
        'row_idx': idx,
        'n_rows': float(n),
        'rel_pos': idx / denom,
        'distance_from_known': distance_from_known,
        'distance_from_known_frac': distance_from_known / denom,
        'hidden_flag': hidden_flag,
        'md': md,
        'md_rel': (md - md.min()) / (md.max() - md.min()) if md.notna().sum() > 1 and md.max() != md.min() else idx / denom,
        'x_centered': x - x.mean(),
        'y_centered': y - y.mean(),
        'z_centered': z - z.mean(),
        'gr': gr_raw,
        'gr_interp': gr_interp,
        'gr_missing': gr_raw.isna().astype('int8'),
        'gr_centered': gr_interp - gr_interp.mean(),
        'carry_tvt': carry,
        'tvt_recent_slope': recent_slope,
        'tvt_recent_volatility': recent_volatility,
    })

    out = add_rolling_features(out, gr_interp, 'gr')
    out = add_rolling_features(out, carry, 'carry_tvt')
    out = add_typewell_dwt_features(out, gr_interp, carry, typewell_lookup.get(well))
    return clean_numeric_frame(out)


def make_masked_frame(df, tail_fraction):
    tvt_col = get_column(df, 'TVT')
    if tvt_col is None:
        return None, None, None
    y_true = numeric_col(df, 'TVT').reset_index(drop=True)
    eval_start = int(len(df) * (1 - tail_fraction))
    masked = df.copy().reset_index(drop=True)
    masked['TVT_input'] = y_true.copy()
    masked.loc[eval_start:, 'TVT_input'] = np.nan
    return masked, y_true, eval_start


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype='float64')
    y_pred = np.asarray(y_pred, dtype='float64')
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    return float(np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))) if mask.any() else np.nan


def select_feature_columns(frame):
    return [col for col in frame.columns if col not in NON_FEATURE_COLUMNS]


def align_feature_frame(frame):
    aligned = frame.reindex(columns=FEATURE_COLUMNS, fill_value=0.0).copy()
    aligned = aligned.replace([np.inf, -np.inf], np.nan)
    return aligned.fillna(0.0)


if train_files:
    sample_path = train_files[0]
    sample_well = well_name_from_horizontal_path(sample_path)
    sample_df = pd.read_csv(sample_path)
    masked_df, y_true, eval_start = make_masked_frame(sample_df, tail_fraction=0.30)
    sample_features = build_features(masked_df, sample_well, train_typewell_lookup)
    print('sample well:', sample_well)
    print('sample feature shape:', sample_features.shape)
    display(sample_features.filter(regex='well|dwt|typewell|carry|distance').head())


## 3. Build Validation Tables

Validation is held out by well and uses masked tail intervals. This mirrors the public task better than random row splits.


In [ ]:
TAIL_FRACTIONS = (0.20, 0.30, 0.40)
MAX_TRAIN_WELLS = 520
MAX_VALIDATION_WELLS = 160
MAX_ROWS_PER_WELL_FOLD = 900
VALIDATION_WELL_FRACTION = 0.20


def sample_eval_rows(features, y_true, eval_start, max_rows, random_state):
    eval_idx = np.arange(eval_start, len(features))
    if len(eval_idx) > max_rows:
        rng = np.random.default_rng(random_state)
        eval_idx = np.sort(rng.choice(eval_idx, size=max_rows, replace=False))
    sampled = features.iloc[eval_idx].copy()
    sampled['target_tvt'] = y_true.iloc[eval_idx].to_numpy()
    sampled['target_residual'] = sampled['target_tvt'] - sampled['carry_tvt']
    return sampled


def build_modeling_table(files, typewell_lookup, max_wells, tail_fractions, max_rows_per_fold, seed=RANDOM_STATE):
    frames = []
    for well_number, path in enumerate(files[:max_wells]):
        well = well_name_from_horizontal_path(path)
        df = pd.read_csv(path)
        for frac_number, tail_fraction in enumerate(tail_fractions):
            masked, y_true, eval_start = make_masked_frame(df, tail_fraction)
            if masked is None:
                continue
            features = build_features(masked, well, typewell_lookup)
            sampled = sample_eval_rows(
                features,
                y_true,
                eval_start,
                max_rows=max_rows_per_fold,
                random_state=seed + 1000 * well_number + frac_number,
            )
            sampled['tail_fraction'] = tail_fraction
            frames.append(sampled)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


rng = np.random.default_rng(RANDOM_STATE)
train_paths = np.array(train_files[:MAX_TRAIN_WELLS], dtype=object)
rng.shuffle(train_paths)
valid_size = max(1, int(len(train_paths) * VALIDATION_WELL_FRACTION))
valid_paths = list(train_paths[:valid_size])
model_train_paths = list(train_paths[valid_size:])

train_table = build_modeling_table(model_train_paths, train_typewell_lookup, MAX_TRAIN_WELLS, TAIL_FRACTIONS, MAX_ROWS_PER_WELL_FOLD)
valid_table = build_modeling_table(valid_paths, train_typewell_lookup, MAX_VALIDATION_WELLS, TAIL_FRACTIONS, MAX_ROWS_PER_WELL_FOLD)
FEATURE_COLUMNS = select_feature_columns(train_table)

print('model train wells:', len(model_train_paths))
print('validation wells:', len(valid_paths))
print('train table:', train_table.shape)
print('valid table:', valid_table.shape)
print('feature count:', len(FEATURE_COLUMNS))
display(train_table.head())


## 4. Train DWT Residual Model

The model predicts residuals over carry-forward. Validation chooses a residual shrinkage factor, because public test has only three wells and raw residual corrections can overfit local validation.


In [ ]:
def fit_dwt_model(train_table):
    X = align_feature_frame(train_table)
    y = train_table['target_residual']
    try:
        model = HistGradientBoostingRegressor(
            loss='squared_error',
            learning_rate=0.04,
            max_iter=320,
            max_leaf_nodes=31,
            min_samples_leaf=40,
            l2_regularization=0.10,
            random_state=RANDOM_STATE,
        )
        model.fit(X, y)
        model_name = 'HistGradientBoostingRegressor'
    except Exception as exc:
        print('HistGradientBoostingRegressor failed; falling back to RandomForestRegressor:', repr(exc))
        model = RandomForestRegressor(
            n_estimators=180,
            max_depth=12,
            min_samples_leaf=20,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        model.fit(X, y)
        model_name = 'RandomForestRegressor'
    return model, model_name


model, model_name = fit_dwt_model(train_table)
print('model:', model_name)

valid_pred_residual = model.predict(align_feature_frame(valid_table))
carry_values = valid_table['carry_tvt'].to_numpy(dtype='float64')
carry_rmse = rmse(valid_table['target_tvt'], valid_table['carry_tvt'])

shrinkage_rows = []
for shrinkage in RESIDUAL_SHRINKAGES:
    pred = carry_values + shrinkage * valid_pred_residual
    score = rmse(valid_table['target_tvt'], pred)
    shrinkage_rows.append({
        'residual_shrinkage': shrinkage,
        'dwt_alignment_rmse': score,
        'delta_vs_carry': score - carry_rmse,
    })

shrinkage_summary = pd.DataFrame(shrinkage_rows).sort_values('dwt_alignment_rmse')
selected_shrinkage = float(shrinkage_summary.iloc[0]['residual_shrinkage'])
dwt_rmse = float(shrinkage_summary.iloc[0]['dwt_alignment_rmse'])
valid_dwt_pred = carry_values + selected_shrinkage * valid_pred_residual

print('carry_forward validation RMSE:', carry_rmse)
print('dwt_alignment validation RMSE:', dwt_rmse)
print('delta RMSE:', dwt_rmse - carry_rmse)
print('selected residual shrinkage:', selected_shrinkage)
display(shrinkage_summary)

valid_scored = valid_table[['well', 'tail_fraction', 'target_tvt', 'carry_tvt']].copy()
valid_scored['dwt_alignment'] = valid_dwt_pred
valid_summary = (
    valid_scored
    .groupby(['tail_fraction'])
    .apply(lambda x: pd.Series({
        'carry_rmse': rmse(x['target_tvt'], x['carry_tvt']),
        'dwt_alignment_rmse': rmse(x['target_tvt'], x['dwt_alignment']),
        'rows': len(x),
        'wells': x['well'].nunique(),
    }))
    .reset_index()
)
valid_summary['delta'] = valid_summary['dwt_alignment_rmse'] - valid_summary['carry_rmse']
display(valid_summary)

use_dwt_model = bool(dwt_rmse < carry_rmse)
print('use_dwt_model_for_submission:', use_dwt_model)


## 5. Inspect DWT Signal

This quick check confirms whether DWT/typewell features are among the strongest residual predictors.


In [ ]:
if hasattr(model, 'feature_importances_'):
    importance = pd.DataFrame({
        'feature': FEATURE_COLUMNS,
        'importance': model.feature_importances_,
    }).sort_values('importance', ascending=False)
else:
    corr_rows = []
    residual = train_table['target_residual'].to_numpy(dtype='float64')
    for feature in FEATURE_COLUMNS:
        values = train_table[feature].to_numpy(dtype='float64')
        if np.nanstd(values) == 0 or np.nanstd(residual) == 0:
            corr = 0.0
        else:
            corr = float(np.corrcoef(values, residual)[0, 1])
        corr_rows.append({'feature': feature, 'abs_correlation_with_residual': abs(corr), 'correlation_with_residual': corr})
    importance = pd.DataFrame(corr_rows).sort_values('abs_correlation_with_residual', ascending=False)

display(importance.head(25))
display(importance[importance['feature'].str.contains('dwt|typewell', case=False, regex=True)].head(25))


## 6. Refit And Generate Submission

If validation improves, refit on more training wells and write `submission.csv`. If not, submit carry-forward as a safe fallback.


In [ ]:
if use_dwt_model:
    refit_table = build_modeling_table(
        train_files,
        train_typewell_lookup,
        max_wells=min(len(train_files), 720),
        tail_fractions=TAIL_FRACTIONS,
        max_rows_per_fold=MAX_ROWS_PER_WELL_FOLD,
    )
    FEATURE_COLUMNS = select_feature_columns(refit_table)
    model, model_name = fit_dwt_model(refit_table)
    print('refit model:', model_name)
    print('refit table:', refit_table.shape)
else:
    print('Validation did not beat carry-forward. Submission will use carry-forward fallback.')


def predict_test_well(df, well):
    features = build_features(df.reset_index(drop=True), well, test_typewell_lookup)
    carry = features['carry_tvt'].to_numpy(dtype='float64')
    if use_dwt_model:
        residual = model.predict(align_feature_frame(features))
        pred = carry + selected_shrinkage * residual
        known = tvt_input_series(df).notna().to_numpy()
        pred[known] = tvt_input_series(df).to_numpy()[known]
        return pd.Series(pred, index=range(len(df)), dtype='float64')
    return pd.Series(carry, index=range(len(df)), dtype='float64')


test_lookup = {well_name_from_horizontal_path(path): path for path in test_files}
well_predictions = {}
for well, path in test_lookup.items():
    df = pd.read_csv(path)
    well_predictions[well] = predict_test_well(df, well).reset_index(drop=True)

fallback = 0.0
non_empty = [pred.dropna().to_numpy() for pred in well_predictions.values() if pred.dropna().size]
if non_empty:
    fallback = float(np.nanmedian(np.concatenate(non_empty)))

submission = sample_submission[[id_col]].copy()
values = []
missing_wells = set()
for sub_id in sample_submission[id_col]:
    well, row_idx = parse_submission_id(sub_id)
    pred = well_predictions.get(well)
    if pred is None or len(pred) == 0:
        missing_wells.add(well)
        values.append(fallback)
    elif 0 <= row_idx < len(pred):
        values.append(float(pred.iloc[row_idx]))
    else:
        values.append(float(pred.iloc[-1]))

submission[target_col] = values
submission.to_csv(SUBMISSION_PATH, index=False)

print('wrote:', SUBMISSION_PATH)
print('selected submission model:', 'dwt_alignment' if use_dwt_model else 'carry_forward')
print('rows:', len(submission))
print('missing wells:', len(missing_wells))
display(submission.head())
display(submission[target_col].describe())


## 7. Readout

Use this as the DWT-inspired candidate after Advanced Modeling V3 dropped to `15.306`.

Compare against:

- current best: Advanced Modeling V2 at `15.049`;
- target direction from public DWT-style notebooks: substantially lower RMSE if multi-scale log-shape alignment transfers to the public wells.

If this notebook improves, keep expanding the DWT idea with true PyWavelets coefficients or direct window matching. If it does not, the failure mode is still useful: it means the public gain may require more direct alignment/reconstruction rather than adding DWT features to the residual tree.
